In [ ]:
# Standard imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
from pathlib import Path

# ML imports
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, classification_report
import mlflow
import mlflow.sklearn

# Create models directory
Path('models').mkdir(parents=True, exist_ok=True)

In [ ]:
# Data acquisition: download and load UCI Heart Disease dataset (Cleveland)
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)
CSV_PATH = DATA_DIR / 'heart.csv'

if not CSV_PATH.exists():
    url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data'
    print('Downloading dataset from UCI...')
    df = pd.read_csv(url, header=None)
    df.columns = ['age','sex','cp','trestbps','chol','fbs','restecg','thalach','exang','oldpeak','slope','ca','thal','target']
    df['target'] = df['target'].apply(lambda x: 1 if x > 0 else 0)
    df.to_csv(CSV_PATH, index=False)
else:
    df = pd.read_csv(CSV_PATH)

print('Loaded shape:', df.shape)
df.head()

In [ ]:
# Quick EDA: class balance, histograms, correlation heatmap
print(df['target'].value_counts(normalize=True))
plt.figure(figsize=(6,4))
sns.countplot(x='target', data=df)
plt.title('Class Balance (0 = No disease, 1 = Disease)')
plt.show()

plt.figure(figsize=(12,8))
sns.heatmap(df.corr(), annot=False, cmap='coolwarm', center=0)
plt.title('Correlation Matrix')
plt.show()

# Histograms for numerical features
num_cols = ['age','trestbps','chol','thalach','oldpeak']
df[num_cols].hist(bins=20, figsize=(12,6))
plt.tight_layout()
plt.show()

In [ ]:
# Prepare features and preprocessing pipeline
feature_cols = ['age','sex','cp','trestbps','chol','fbs','restecg','thalach','exang','oldpeak','slope','ca','thal']
X = df[feature_cols].copy()
y = df['target'].copy()

numeric_features = ['age','trestbps','chol','thalach','oldpeak']
categorical_features = [c for c in feature_cols if c not in numeric_features]

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

# Train / test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print('Train shape', X_train.shape, 'Test shape', X_test.shape)

In [ ]:
# Model training and MLflow logging
mlflow.set_experiment('heart_disease_classification')
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'logreg': LogisticRegression(max_iter=1000, solver='liblinear'),
    'rf': RandomForestClassifier(n_estimators=200, random_state=42)
}

best_model = None
best_auc = 0

for name, estimator in models.items():
    with mlflow.start_run(run_name=name):
        pipe = Pipeline([('preprocessor', preprocessor), ('clf', estimator)])
        scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='roc_auc')
        mean_auc = float(np.mean(scores))
        mlflow.log_param('model', name)
        mlflow.log_metric('cv_roc_auc', mean_auc)
        pipe.fit(X_train, y_train)
        preds = pipe.predict(X_test)
        proba = pipe.predict_proba(X_test)[:,1] if hasattr(pipe, 'predict_proba') else None
        acc = accuracy_score(y_test, preds)
        prec = precision_score(y_test, preds)
        rec = recall_score(y_test, preds)
        auc = roc_auc_score(y_test, proba) if proba is not None else 0
        mlflow.log_metric('test_accuracy', float(acc))
        mlflow.log_metric('test_precision', float(prec))
        mlflow.log_metric('test_recall', float(rec))
        mlflow.log_metric('test_roc_auc', float(auc))
        cr = classification_report(y_test, preds, output_dict=True)
        with open('classification_report_{}.json'.format(name), 'w') as fh:
            json.dump(cr, fh)
        mlflow.log_artifact('classification_report_{}.json'.format(name))
        mlflow.sklearn.log_model(pipe, 'model')
        print(f'{name} - CV ROC-AUC: {mean_auc:.4f} - Test AUC: {auc:.4f} - Acc: {acc:.4f}')
        if auc > best_auc:
            best_auc = auc
            best_model = pipe

# Save final preprocessor and best model locally for serving
joblib.dump(best_model, 'models/heart_model.pkl')
meta = {'feature_order': feature_cols, 'numeric_features': numeric_features, 'categorical_features': categorical_features}
with open('models/pipeline_metadata.json', 'w') as fh:
    json.dump(meta, fh)
print('Saved best model to models/heart_model.pkl')
print('Pipeline metadata saved to models/pipeline_metadata.json')

In [ ]:
# Quick inference example using the saved model
model = joblib.load('models/heart_model.pkl')
sample = X_test.iloc[0:3]
preds = model.predict(sample)
probs = model.predict_proba(sample)[:,1] if hasattr(model, 'predict_proba') else None
print('Preds:', preds)
print('Probs:', probs)

## Notes
- The notebook logs experiments with MLflow and saves the final scikit-learn Pipeline as `models/heart_model.pkl`.
- `models/pipeline_metadata.json` captures feature ordering for the Flask app to reproduce preprocessing consistently.
- For deployment to Hugging Face Model Hub, upload the `models/` contents to a repo and set the Flask config to load from that repo or mount `models/` into the container.